# Parser les PVs I1
Génère à partir de `../data/raw/I1/I1_1.xlsx`, `I1_2.xlsx`, `I1_3.xlsx` :
- `matieres.csv`
- `parcours.csv`
- `notes.csv`

In [ ]:
import pandas as pd
import numpy as np
import re
import os

RAW_DIR    = '../data/raw/I1'
PARSED_DIR = '../data/parsed'
os.makedirs(PARSED_DIR, exist_ok=True)

PROMO = 'I1'

# Mapping id_matière → semestre pour I1
SEMESTRE_MAP = {
    'S1': ['i01','i02','i03','i04','i05','i06','i07','i08','i09','i10','i11','i12','i13'],
    'S2': ['i14','i15','i16','i17','i18','i19','i20','i21','i22','i23','i24','i25','i26']
}
ID_TO_SEM = {mid: sem for sem, ids in SEMESTRE_MAP.items() for mid in ids}

# Fichiers PV
PV_FILES = [
    os.path.join(RAW_DIR, 'I1_1.xlsx'),
    os.path.join(RAW_DIR, 'I1_2.xlsx'),
    os.path.join(RAW_DIR, 'I1_3.xlsx'),
]

## 1. Fonctions utilitaires

In [ ]:
def parse_note(val):
    """Convertit une note en float (gère '12,50', 12.5, NaN)."""
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        val = val.replace(',', '.').strip()
        try:
            return float(val)
        except:
            return np.nan
    try:
        return float(val)
    except:
        return np.nan


def extract_mat_id(code):
    """
    Extrait l'id matière depuis un code de sous-colonne.
    'ex_i01' -> 'i01' | 'mg_i14' -> 'i14' | 'moy_ue' -> None
    """
    if isinstance(code, str):
        m = re.search(r'(i\d{2})$', code)
        if m:
            return m.group(1)
    return None


def is_unnamed(val):
    """Vérifie si une valeur est une colonne 'Unnamed' ou vide (cellule fusionnée)."""
    if pd.isna(val):
        return True
    if isinstance(val, str) and (val.strip() == '' or val.startswith('Unnamed:')):
        return True
    return False

## 2. Parser la structure d'un PV

In [ ]:
def parse_pv(filepath):
    """
    Parse un fichier PV Excel.

    Structure attendue dans Excel :
      Ligne 0 : N°, Nom, Prenom, [nom_mat (fusionné)], [nom_module], ..., moy_s1, moy_s2, mann, rang, rann
      Ligne 1 : (vide), (vide), (vide), ex_iXX, cc_iXX, mg_iXX, r_iXX, moy_ue, ...
      Lignes 2+ : données étudiants

    Retourne :
      df_students : DataFrame avec colonnes nommées explicitement
      structure   : liste de dicts décrivant chaque matière/module trouvé
                    [
                      {
                        'id_mat'      : 'i01',
                        'nom_mat'     : 'algo.num1',
                        'nom_module'  : 'Mathématiques 5',
                        'col_mg'      : 5,           # index colonne mg_iXX
                        'col_mod_moy' : 11,          # index colonne moy_ue du module
                      },
                      ...
                    ]
    """
    df_raw = pd.read_excel(filepath, header=None)

    # --- Lignes d'en-tête ---
    row0 = df_raw.iloc[0].tolist()   # noms matières/modules (avec NaN pour les fusionnées)
    row1 = df_raw.iloc[1].tolist()   # codes sous-colonnes

    # Forward-fill row0 pour propager le nom de matière sur les Unnamed (cellules fusionnées)
    row0_filled = []
    last_named = None
    for v in row0:
        if not is_unnamed(v):
            last_named = v
        row0_filled.append(last_named)

    # --- Données étudiants ---
    df_data = df_raw.iloc[2:].reset_index(drop=True).copy()
    n_cols = len(df_data.columns)

    # --- Identifier les colonnes fixes (identité + résultats annuels) ---
    # On cherche dans row1 les codes connus
    result_cols = {}   # nom_logique -> index colonne
    identity_cols = {} # 'N°','Nom','Prenom' -> index

    RESULT_CODES = {'moy_s1', 'moy_s2', 'mann', 'rang', 'rann'}
    IDENTITY_CODES = {'n°', 'nom', 'prenom'}  # on comparera en minuscule

    for i, (h0, h1) in enumerate(zip(row0_filled, row1)):
        h1_str = str(h1).strip().lower() if not pd.isna(h1) else ''
        h0_str = str(h0).strip().lower() if not pd.isna(h0) else ''

        if h0_str in IDENTITY_CODES:
            identity_cols[str(h0).strip()] = i
        elif h1_str in RESULT_CODES:
            result_cols[h1_str] = i
        elif h0_str in RESULT_CODES:  # parfois dans row0
            result_cols[h0_str] = i

    # --- Identifier matières et modules ---
    # On itère sur les colonnes et on cherche les mg_iXX et moy_ue
    # Un module est identifié par une colonne dont row1 = 'moy_ue'
    # Sa valeur row0 = nom du module

    # Construire un mapping col_index -> (header0, header1)
    col_info = [(i, row0_filled[i], str(row1[i]).strip() if not pd.isna(row1[i]) else '') 
                for i in range(n_cols)]

    # Identifier les colonnes moy_ue (= colonne de moyenne de module)
    module_moy_cols = {}  # col_index -> nom_module
    for i, h0, h1 in col_info:
        if h1.lower() == 'moy_ue':
            module_moy_cols[i] = str(h0).strip() if not pd.isna(h0) else ''

    # Identifier les colonnes mg_iXX (= note de matière)
    mg_cols = []  # liste de (col_index, id_mat, nom_mat)
    for i, h0, h1 in col_info:
        if re.match(r'^mg_i\d{2}$', h1.lower()):
            mat_id = extract_mat_id(h1)
            nom_mat = str(h0).strip() if not pd.isna(h0) else ''
            mg_cols.append((i, mat_id, nom_mat))

    # Associer chaque matière à son module :
    # Le module d'une matière est la colonne moy_ue qui suit le groupe de cette matière.
    # On cherche le prochain moy_ue après chaque mg_iXX.
    module_moy_positions = sorted(module_moy_cols.keys())

    structure = []
    for (col_mg, mat_id, nom_mat) in mg_cols:
        # Module = premier moy_ue dont la position est >= col_mg
        col_mod = next((p for p in module_moy_positions if p >= col_mg), None)
        nom_module = module_moy_cols.get(col_mod, '') if col_mod is not None else ''
        structure.append({
            'id_mat'     : mat_id,
            'nom_mat'    : nom_mat,
            'nom_module' : nom_module,
            'col_mg'     : col_mg,
            'col_mod_moy': col_mod,
        })

    return df_data, structure, identity_cols, result_cols


# --- Test rapide ---
for f in PV_FILES:
    df_data, structure, identity_cols, result_cols = parse_pv(f)
    print(f'\n=== {os.path.basename(f)} ===')
    print(f'  Étudiants : {len(df_data)}')
    print(f'  Matières  : {len(structure)}')
    print(f'  identity_cols : {identity_cols}')
    print(f'  result_cols   : {result_cols}')
    for s in structure:
        print(f"    {s['id_mat']:5s}  {s['nom_mat']:35s}  module: {s['nom_module']}  sem: {ID_TO_SEM.get(s['id_mat'],'?')}")

## 3. Construire `matieres.csv`
Colonnes : `id_mat | nom_mat | nom_module | semestre`

On agrège les infos des 3 PVs (déduplication par id_mat).

In [ ]:
matieres_dict = {}  # id_mat -> dict

for f in PV_FILES:
    _, structure, _, _ = parse_pv(f)
    for s in structure:
        mid = s['id_mat']
        if mid not in matieres_dict:
            matieres_dict[mid] = {
                'id_mat'    : mid,
                'nom_mat'   : s['nom_mat'],
                'nom_module': s['nom_module'],
                'semestre'  : ID_TO_SEM.get(mid, '?'),
            }

df_matieres = pd.DataFrame(list(matieres_dict.values()))
df_matieres = df_matieres.sort_values('id_mat').reset_index(drop=True)

# Exporter
out_matieres = os.path.join(PARSED_DIR, 'matieres.csv')
df_matieres.to_csv(out_matieres, index=False, encoding='utf-8-sig')
print(f'matieres.csv exporté : {len(df_matieres)} matières')
df_matieres

## 4. Construire `parcours.csv` et `notes.csv`

Pour identifier les étudiants, on utilise `mapping_id_nom.csv`.

In [ ]:
# Charger le mapping id <-> nom/prénom
mapping_path = os.path.join(PARSED_DIR, 'mapping_id_nom.csv')
df_mapping = pd.read_csv(mapping_path, encoding='utf-8-sig')
print(f'Mapping chargé : {len(df_mapping)} étudiants')
print(df_mapping.head())
print(df_mapping.columns.tolist())

In [ ]:
def normalize_name(s):
    """Normalise un nom/prénom pour la jointure."""
    if pd.isna(s):
        return ''
    return str(s).strip().upper().replace('-', ' ')


def get_student_id(nom, prenom, df_mapping):
    """
    Cherche l'id étudiant dans le mapping par (Nom, Prenom).
    Retourne l'id ou None si introuvable.
    Suppose que df_mapping a des colonnes 'nom', 'prenom', 'id' (adapter si besoin).
    """
    nom_n    = normalize_name(nom)
    prenom_n = normalize_name(prenom)

    # Colonnes du mapping (adapter les noms si nécessaire)
    col_nom    = [c for c in df_mapping.columns if 'nom' in c.lower() and 'pre' not in c.lower()][0]
    col_prenom = [c for c in df_mapping.columns if 'pre' in c.lower()][0]
    col_id     = [c for c in df_mapping.columns if 'id' in c.lower()][0]

    mask = (
        df_mapping[col_nom].apply(normalize_name) == nom_n
    ) & (
        df_mapping[col_prenom].apply(normalize_name) == prenom_n
    )
    matches = df_mapping[mask]
    if len(matches) == 1:
        return matches.iloc[0][col_id]
    elif len(matches) > 1:
        print(f'  [WARN] Plusieurs correspondances pour {nom} {prenom}')
        return matches.iloc[0][col_id]
    else:
        print(f'  [WARN] Aucune correspondance pour {nom} {prenom}')
        return None

In [ ]:
all_parcours = []
all_notes    = []

# Pour éviter les doublons dans parcours (un étudiant peut apparaître dans plusieurs PVs)
seen_students = set()
# Pour éviter les doublons dans notes (un étudiant + une matière ne doit apparaître qu'une fois)
seen_notes = set()

for f in PV_FILES:
    print(f'\n=== Traitement : {os.path.basename(f)} ===')
    df_data, structure, identity_cols, result_cols = parse_pv(f)

    # Colonnes identité
    col_nom    = identity_cols.get('Nom')    or identity_cols.get('nom')
    col_prenom = identity_cols.get('Prenom') or identity_cols.get('prenom')

    for _, row in df_data.iterrows():
        nom    = row[col_nom]    if col_nom    is not None else None
        prenom = row[col_prenom] if col_prenom is not None else None

        # Ignorer les lignes vides
        if pd.isna(nom) or str(nom).strip() == '':
            continue

        # Récupérer l'id étudiant
        student_id = get_student_id(nom, prenom, df_mapping)

        # --- PARCOURS ---
        if student_id is not None and student_id not in seen_students:
            moy_s1 = parse_note(row[result_cols['moy_s1']]) if 'moy_s1' in result_cols else np.nan
            moy_s2 = parse_note(row[result_cols['moy_s2']]) if 'moy_s2' in result_cols else np.nan
            moy_ann = parse_note(row[result_cols['mann']])  if 'mann'   in result_cols else np.nan
            rang   = parse_note(row[result_cols['rang']])   if 'rang'   in result_cols else np.nan
            rann   = str(row[result_cols['rann']]).strip()  if 'rann'   in result_cols else ''

            # Passage : 'A' = Admis, 'R' = Rattrapage
            passage_map = {'A': 'Admis', 'R': 'Rattrapage'}
            passage = passage_map.get(rann.upper(), rann)

            all_parcours.append({
                'id'      : student_id,
                'promo'   : PROMO,
                'moy_s1'  : moy_s1,
                'moy_s2'  : moy_s2,
                'moy_ann' : moy_ann,
                'rang'    : rang,
                'passage' : passage,
            })
            seen_students.add(student_id)

        # --- NOTES ---
        for s in structure:
            mat_id  = s['id_mat']
            col_mg  = s['col_mg']
            col_mod = s['col_mod_moy']

            key = (student_id, mat_id)
            if student_id is None or key in seen_notes:
                continue

            mg_mat = parse_note(row[col_mg])
            mg_mod = parse_note(row[col_mod]) if col_mod is not None else np.nan

            all_notes.append({
                'id'        : student_id,
                'promo'     : PROMO,
                'id_mat'    : mat_id,
                'mg_mat'    : mg_mat,
                'mg_module' : mg_mod,
            })
            seen_notes.add(key)

print(f'\nTotal parcours : {len(all_parcours)}')
print(f'Total notes    : {len(all_notes)}')

## 5. Export des fichiers CSV

In [ ]:
# --- parcours.csv ---
df_parcours = pd.DataFrame(all_parcours)
df_parcours = df_parcours.sort_values('id').reset_index(drop=True)

out_parcours = os.path.join(PARSED_DIR, 'parcours.csv')
df_parcours.to_csv(out_parcours, index=False, encoding='utf-8-sig')
print(f'parcours.csv exporté : {len(df_parcours)} étudiants')
df_parcours.head()

In [ ]:
# --- notes.csv ---
df_notes = pd.DataFrame(all_notes)
df_notes = df_notes.sort_values(['id', 'id_mat']).reset_index(drop=True)

out_notes = os.path.join(PARSED_DIR, 'notes.csv')
df_notes.to_csv(out_notes, index=False, encoding='utf-8-sig')
print(f'notes.csv exporté : {len(df_notes)} lignes')
print(f'  (~{len(df_notes) // max(len(df_parcours),1)} matières/étudiant en moyenne)')
df_notes.head(10)

## 6. Vérifications

In [ ]:
print('=== matieres.csv ===')
print(df_matieres.groupby('semestre')['id_mat'].count().to_string())
print()

print('=== parcours.csv ===')
print(f"Admis      : {(df_parcours['passage']=='Admis').sum()}")
print(f"Rattrapage : {(df_parcours['passage']=='Rattrapage').sum()}")
print(f"Autre      : {(~df_parcours['passage'].isin(['Admis','Rattrapage'])).sum()}")
print()

print('=== notes.csv ===')
print(f"Matières uniques : {df_notes['id_mat'].nunique()}")
print(f"Étudiants uniques : {df_notes['id'].nunique()}")
print(f"Doublons (id, id_mat) : {df_notes.duplicated(['id','id_mat']).sum()}")

# Vérifier que chaque étudiant a bien 26 matières
notes_per_student = df_notes.groupby('id')['id_mat'].count()
print(f"\nNotes par étudiant (min/max/mean) : {notes_per_student.min()} / {notes_per_student.max()} / {notes_per_student.mean():.1f}")
if (notes_per_student < 26).any():
    print('[WARN] Certains étudiants ont moins de 26 matières :')
    print(notes_per_student[notes_per_student < 26])